In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
df = pd.read_csv('online_retail_II.csv')

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


In [17]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [18]:
df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [19]:
df = df.dropna(subset=['Customer ID', 'Description'])
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

In [20]:
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

product_sales = df.groupby(['Description', 'YearMonth'])['Quantity'].sum().reset_index()
product_sales['YearMonth'] = product_sales['YearMonth'].astype(str)

In [21]:
produk_pilih = "PINK CHERRY LIGHTS"

data = product_sales[
    product_sales['Description'].str.contains(produk_pilih, case=False, na=False)
].copy()

In [22]:
data['t'] = np.arange(len(data))

data['month'] = pd.to_datetime(data['YearMonth']).dt.month
data['year'] = pd.to_datetime(data['YearMonth']).dt.year

In [23]:
for i in range(1, 6):
    data[f'lag{i}'] = data['Quantity'].shift(i)

data = data.dropna()

print("\nData setelah feature engineering:")
print(data.head())


Data setelah feature engineering:
                    Description YearMonth  Quantity  t  month  year   lag1  \
29817  LIGHT PINK CHERRY LIGHTS   2010-08        72  5      8  2010  185.0   
29818  LIGHT PINK CHERRY LIGHTS   2010-10         6  6     10  2010   72.0   
38271        PINK CHERRY LIGHTS   2009-12       786  7     12  2009    6.0   
38272        PINK CHERRY LIGHTS   2010-01       299  8      1  2010  786.0   
38273        PINK CHERRY LIGHTS   2010-02       258  9      2  2010  299.0   

        lag2   lag3   lag4   lag5  
29817  316.0  165.0  275.0  401.0  
29818  185.0  316.0  165.0  275.0  
38271   72.0  185.0  316.0  165.0  
38272    6.0   72.0  185.0  316.0  
38273  786.0    6.0   72.0  185.0  


In [24]:
fitur = ['t', 'month', 'year'] + [f'lag{i}' for i in range(1,6)]

X = data[fitur]
y = data['Quantity']

In [25]:
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X, y)

data['Prediksi'] = model.predict(X)

In [26]:
mae = mean_absolute_error(y, data['Prediksi'])
mse = mean_squared_error(y, data['Prediksi'])
rmse = np.sqrt(mse)
r2 = r2_score(y, data['Prediksi'])

print("\n=== EVALUASI MODEL ===")
print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")


=== EVALUASI MODEL ===
MAE  : 86.13
MSE  : 10061.18
RMSE : 100.31
R2   : 0.8492


In [ ]:
product_name = 'WHITE HANGING HEART T-LIGHT HOLDER'
future_rows = []

last = data.iloc[-1].copy()

for i in range(3):
    new_t = last['t'] + 1
    new_month = (last['month'] % 12) + 1
    new_year = last['year'] + (1 if new_month == 1 else 0)

    new_data = {
        't': new_t,
        'month': new_month,
        'year': new_year
    }

    # isi lag dinamis
    for j in range(1, 6):
        if j == 1:
            new_data[f'lag{j}'] = last['Quantity']
        else:
            new_data[f'lag{j}'] = last[f'lag{j-1}']

    new_df = pd.DataFrame([new_data])

    pred = model.predict(new_df)[0]

    future_rows.append({
        'Description': product_name,  
        'Year': int(new_year),
        'Month': int(new_month),
        'Predicted_Quantity': int(pred)
    })

    
    for j in range(5, 1, -1):
        last[f'lag{j}'] = last[f'lag{j-1}']

    last['lag1'] = last['Quantity']
    last['Quantity'] = pred
    last['t'] = new_t
    last['month'] = new_month
    last['year'] = new_year


result_df = pd.DataFrame(future_rows)

print("\n=== DATA HASIL PREDIKSI ===")
print(result_df)


=== DATA HASIL PREDIKSI ===
                          Description  Year  Month  Predicted_Quantity
0  WHITE HANGING HEART T-LIGHT HOLDER  2011      2                 180
1  WHITE HANGING HEART T-LIGHT HOLDER  2011      3                 166
2  WHITE HANGING HEART T-LIGHT HOLDER  2011      4                 176


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Prediksi
y_pred = model.predict(X)

# Evaluasi
mae = mean_absolute_error(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y, y_pred)

# =========================================
# HASIL EVALUASI
# =========================================
print("\nEVALUASI MODEL ")
print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2   : {r2:.4f}")


EVALUASI MODEL 
MAE  : 86.13
MSE  : 10061.18
RMSE : 100.31
R2   : 0.8492


In [29]:
import pickle

model_data = {
    'model': model,
    'fitur': ['t','month','year','lag1','lag2','lag3','lag4','lag5']
}

with open('model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("✅ Model + fitur berhasil disimpan")

✅ Model + fitur berhasil disimpan
